# 🌸 鳶尾花隨機森林分類器 API 教學：實作 `/train` 線上訓練與模型更新端點

> 對象：修習隨機森林分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 145~168 行的 `train_api` 函數與 `train_save.py` 的 `train_and_save_model` 函數
>
> 這份 Notebook 會帶你**一步步拆解**線上模型重新訓練與即時更新機制，並用 `TestClient` 模擬完整呼叫流程。

## 🎯 學習目標

- [ ] 理解「線上重訓 (Online Retraining)」的概念與應用場景
- [ ] 看懂 FastAPI 訓練端點的請求與回應模型（`TrainConfig` 與 `TrainResult`）
- [ ] 掌握 `train_and_save_model()` 如何依超參數（`n_estimators`, `max_depth`, `test_size`, `random_state`）重新訓練並覆蓋 `.joblib` 模型檔
- [ ] 理解 `load_model_state()` 重新載入全域狀態（`MODEL_STATE`）的必要性
- [ ] 能使用 `TestClient` 發送 POST `/train` 請求，並驗證模型更新後的各項指標與特徵重要性

## 📌 背景：`/train` 端點運作流程

在現實部署環境中，當我們想要嘗試不同的隨機森林超參數（如增加決策樹數量 `n_estimators` 或限制 `max_depth`）、調整測試集切分比例 `test_size` 時，我們希望**不用關閉或重啟伺服器**就能完成模型更新。

`/train` 端點的內部三大核心步驟：

```
[收到 TrainConfig 請求]
       │
       ▼
1. train_and_save_model(...)  ──> 重新訓練隨機森林，覆蓋 iris_model.joblib
       │
       ▼
2. load_model_state()         ──> 重新讀取 joblib 檔，更新全域變數 MODEL_STATE
       │
       ▼
3. 回傳 TrainResult(**res)     ──> 回傳準確度 (Accuracy)、特徵重要性與訓練耗時
```

接下來我們一步步拆解並動手實作。

In [ ]:
# ============================================
# 0. 載入套件與環境設定
# ============================================

import os, sys, time
import joblib
from pprint import pprint

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# 匯入訓練邏輯模組
from train_save import train_and_save_model

print("環境準備完畢，已成功載入 train_and_save_model 模組。")

---
## Part 1：測試模型訓練與序列化儲存 (`train_and_save_model`)

先單獨呼叫 `train_and_save_model` 觀察訓練流程與回傳結果。

In [ ]:
# 執行訓練並檢視回傳字典
result = train_and_save_model(
    n_estimators=100,
    max_depth=5,
    test_size=0.2,
    random_state=42
)

print("\n訓練回傳字典內容：")
pprint(result)

---
## Part 2：定義請求與回應的 Pydantic 模型

在 `app.py` 中，訓練端點需要接收使用者傳遞的超參數，並回傳格式化的訓練結果：

- **`TrainConfig`**（請求模型）：包含 `n_estimators` (10~500)、`max_depth` (0~20)、`test_size` (0.1~0.5) 與 `random_state` (>=0)。
- **`TrainResult`**（回應模型）：回傳 `status`、`accuracy`、`train_time`、`feature_importances` 與 `message`。

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class TrainConfig(BaseModel):
    n_estimators: int = Field(100, description="決策樹數量", ge=10, le=500)
    max_depth: Optional[int] = Field(None, description="最大深度 (None/0 表示無限制)", ge=0, le=20)
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1, le=0.5)
    random_state: int = Field(42, description="隨機種子", ge=0)

class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")
    accuracy: float = Field(..., description="測試集準確度")
    train_time: float = Field(..., description="訓練耗時 (秒)")
    feature_importances: dict[str, float] = Field(..., description="各特徵之重要性分佈")
    message: str = Field(..., description="提示訊息")

print("Pydantic 訓練模型定義完成！")

---
## Part 3：建立 FastAPI 應用與 `/train` 線上重訓端點

建立 FastAPI App 並定義 `/train` 端點，線上重訓後更新 `MODEL_STATE`。

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Iris Training API")
MODEL_STATE = {}

def load_model_state():
    global MODEL_STATE
    model_path = os.path.join(current_dir, "iris_model.joblib")
    if not os.path.exists(model_path):
        train_and_save_model()
    model_data = joblib.load(model_path)
    MODEL_STATE.clear()
    MODEL_STATE.update({
        "model": model_data["model"],
        "target_names": model_data["target_names"],
        "accuracy": model_data.get("accuracy", 0.0),
        "feature_importances": model_data.get("feature_importances", {})
    })
    print(f"狀態重載完成！當前準確度：{MODEL_STATE['accuracy']:.4f}")

# 啟動時先載入一次
load_model_state()

@app.post("/train", response_model=TrainResult)
def train_api(config: TrainConfig):
    try:
        depth_val = None if config.max_depth == 0 or config.max_depth is None else config.max_depth
        res = train_and_save_model(
            n_estimators=config.n_estimators,
            max_depth=depth_val,
            test_size=config.test_size,
            random_state=config.random_state
        )
        load_model_state()
        return TrainResult(**res)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

print("FastAPI /train 端點建立完成！")

---
## Part 4：使用 `TestClient` 模擬呼叫 `/train` 端點

透過 FastAPI 內建的 `TestClient` 發送 POST 請求驗證。

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 發送測試請求 (將樹的數量改為 200，最大深度限制為 3)
payload = {
    "n_estimators": 200,
    "max_depth": 3,
    "test_size": 0.2,
    "random_state": 42
}

response = client.post("/train", json=payload)

print("HTTP 狀態碼：", response.status_code)
print("API 回傳結果：")
pprint(response.json())